In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate

1st Pipeline(Indexing Phase)

In [4]:
loader = PyPDFLoader("../data/College.pdf")
pdf_content = loader.load()

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunk_data = splitter.split_documents(pdf_content)
len(chunk_data)


10

In [9]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [10]:
vector_store = Chroma.from_documents(
    documents=chunk_data,
    embedding=embeddings,
)

2nd Pipeline (Query Phase / retrival phase)

In [16]:
query = "Colleges in Mumbai"
relevant_chunks = vector_store.similarity_search(query = query)

In [18]:
print(len(relevant_chunks))
print(relevant_chunks[0].page_content)

4
With 95 percentile, you may also have a realistic chance at colleges that are generally stronger than
some of the above, depending on the year's cutoﬀs and CAP rounds, such as:
Vidyalankar Institute of Technology
Don Bosco Institute of Technology
Atharva College of Engineering
Xavier Institute of Engineering
If you tell me:
your category (OPEN/OBC/SC/ST/EWS),
whether you're applying through MHT-CET or JEE Main, and
whether you are from Home University (Mumbai University),
I can prepare a complete preference list of 30–40 Mumbai colleges in the exact order you should ﬁll
them in CAP counselling to maximize your chances of getting the best CS/IT seat.
give me in marathi
तुझे  95 percentile (MHT-CET) आ ण  CS/IT, Coding, Software Development, AI Engineer हे  ल य  ल ात  घेऊन
मी  या  कॉलेजांची  तुलना  केली  आहे .
माझी  अं तम  रँ कग  (CS/IT साठी )
रँक कॉलेज रे टग माझे  मत
🥇
1
SIES Graduate School of
Technology, Nerul ⭐⭐⭐⭐⭐
सवा त  चांगला  पया य .  लेसम ट , को डग  क चर , इंड ट ी  ए  पोजर  आ ण

In [21]:
context = ""
for chunk in relevant_chunks:
    context += chunk.page_content + "\n"
    
print(context)

With 95 percentile, you may also have a realistic chance at colleges that are generally stronger than
some of the above, depending on the year's cutoﬀs and CAP rounds, such as:
Vidyalankar Institute of Technology
Don Bosco Institute of Technology
Atharva College of Engineering
Xavier Institute of Engineering
If you tell me:
your category (OPEN/OBC/SC/ST/EWS),
whether you're applying through MHT-CET or JEE Main, and
whether you are from Home University (Mumbai University),
I can prepare a complete preference list of 30–40 Mumbai colleges in the exact order you should ﬁll
them in CAP counselling to maximize your chances of getting the best CS/IT seat.
give me in marathi
तुझे  95 percentile (MHT-CET) आ ण  CS/IT, Coding, Software Development, AI Engineer हे  ल य  ल ात  घेऊन
मी  या  कॉलेजांची  तुलना  केली  आहे .
माझी  अं तम  रँ कग  (CS/IT साठी )
रँक कॉलेज रे टग माझे  मत
🥇
1
SIES Graduate School of
Technology, Nerul ⭐⭐⭐⭐⭐
सवा त  चांगला  पया य .  लेसम ट , को डग  क चर , इंड ट ी  ए  पोजर  आ ण
म

In [22]:
llm = ChatOpenAI(model="gpt-4o")

Chain - context_generate | prompt | llm | strparser


In [31]:
def get_context(query : str):
    relevant_chunks = vector_store.similarity_search(query = query)
    context = ""
    for chunk in relevant_chunks:
        context += chunk.page_content + "\n"
        
    return {
        "context" : context,
        "question" : query
    }

In [32]:
prompt = PromptTemplate.from_template("""
    you are helpful assistant and provide the answer based on the context
    if you dont know the answer then you can say 'I dont know'
    Context:{context}  
    Question:{question}        
""")

In [33]:
rag_chain = get_context | prompt | llm

In [34]:
res = rag_chain.invoke("tell me about usha mittal college")
print(res.content)

Usha Mittal Institute of Technology, primarily a women's engineering institute under SNDT University, has some appealing features but also some challenges. It boasts good infrastructure and reports some high placement packages. However, its consistency in placements and the coding ecosystem are considered weaker compared to some other colleges mentioned. It may not be the top choice if you are prioritizing a strong coding culture and consistent placements, especially in comparison to the likes of SIES GST and Father Agnel, which have been noted for stronger placements and coding cultures.
